# Notebook 01: Natural Language Processing - Text Generation

**Objective:** Learn how to generate coherent text continuations using pretrained transformer models from HuggingFace.

**Prerequisites:** Python basics, familiarity with installing packages. No prior ML experience needed.

**Learning Objectives:**
- Understand how autoregressive text generation works
- Load and use pretrained language models from HuggingFace
- Control generation quality with temperature, top-k, and top-p
- Compare the Pipeline API vs direct model usage
- Recognize where small language models succeed and fail

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|--------------|------------|------|---------|-------------------|-------|
| **CPU (Small)** | distilgpt2 | 82MB | 2GB | 4GB RAM, CPU | Fast, educational |
| **GPU (Medium)** | gpt2-medium | 1.5GB | 4GB | 8GB VRAM (RTX 4080) | Better quality |

### Software Requirements
- Python 3.10+
- Libraries: `transformers`, `torch`, `pandas`
- See `requirements.txt` for full list

## Overview

**Text generation** is the task of producing coherent text based on a given prompt. It is one of the most widely used applications of transformer models, powering tools from code assistants to creative writing aids.

**How autoregressive generation works:**
1. You provide a text prompt (e.g., "Once upon a time")
2. The model predicts a probability distribution over the entire vocabulary for the *next* token
3. A token is selected from that distribution (using a decoding strategy like sampling or beam search)
4. The selected token is appended to the input, and the process repeats

The key insight is that each generated token becomes part of the context for generating the next one. This is why the process is called *autoregressive* -- the model's own outputs feed back into itself.

**Use cases:** Creative writing assistance, code completion, chatbots, content drafting, data augmentation.

## Expected Behaviors

When you run this notebook, here's what you should see:

### First Time Running
- **Model download**: ~82MB for distilgpt2, ~1.5GB for gpt2-medium
- Downloads are cached in `~/.cache/huggingface/hub/` -- subsequent runs skip this step

### Text Generation Outputs
- Generated text should be **grammatically coherent** but may not always be factually accurate
- **Temperature 0.3**: More repetitive, focused text
- **Temperature 0.7**: Balanced creativity and coherence
- **Temperature 1.2**: More random, surprising text

### Performance
- **CPU (distilgpt2)**: ~2-5 seconds per generation
- **GPU (distilgpt2)**: ~0.5-1 second per generation
- **GPU (gpt2-medium)**: ~1-2 seconds per generation

### Troubleshooting
- **"CUDA out of memory"**: Switch to distilgpt2 or restart kernel
- **Slow first run**: Model is downloading; cached runs are much faster
- **Repetitive text**: Increase temperature or adjust top_k/top_p

## Setup and Installation

We begin by importing all required libraries and checking our environment. This cell also sets the random seed so that results are reproducible across runs.

In [ ]:
# All imports in a single cell
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, set_seed

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Version metadata
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Model Selection

We provide two model options. **distilgpt2** is a distilled (compressed) version of GPT-2 that runs comfortably on any CPU. If you have a GPU with 8GB+ VRAM, uncomment the `gpt2-medium` line for noticeably better generation quality.

Both models use the same GPT-2 architecture -- an autoregressive transformer decoder trained on web text. The difference is parameter count: distilgpt2 has 82M parameters vs gpt2-medium's 355M.

In [ ]:
# CHOOSE YOUR MODEL:

# Option 1: CPU-friendly (recommended for beginners)
MODEL_NAME = "distilgpt2"  # 82MB, fast on CPU

# Option 2: GPU-optimized (uncomment if you have RTX 4080 or similar)
# MODEL_NAME = "gpt2-medium"  # 1.5GB, better quality, needs GPU

print(f"Selected model: {MODEL_NAME}")

---

## Method 1: Using Pipeline (Simplest)

The `pipeline` API is HuggingFace's highest-level interface. It wraps tokenization, model inference, and decoding into a single callable object. This is the recommended starting point -- you can generate text in just two lines of code.

Under the hood, the pipeline handles:
- Loading the tokenizer and model
- Converting your text prompt into token IDs
- Running the model's forward pass
- Decoding the output token IDs back into readable text

In [ ]:
# Create a text generation pipeline
try:
    print(f"Loading {MODEL_NAME}...")
    generator = pipeline(
        "text-generation",
        model=MODEL_NAME,
        device=0 if torch.cuda.is_available() else -1
    )
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Make sure you have an internet connection for the first download.")
    print("If behind a firewall, try: pip install transformers[torch] --upgrade")
    raise

### Basic Text Generation

Let's start with the simplest case: provide a prompt and let the model continue it. The `max_length` parameter controls the total length (prompt + generated text), and `temperature` controls how "creative" the output is.

In [ ]:
# Generate text from a prompt
prompt = "Once upon a time in a distant galaxy"

result = generator(
    prompt,
    max_length=50,
    num_return_sequences=1,
    temperature=0.7,
    do_sample=True,
    pad_token_id=generator.tokenizer.eos_token_id
)

print("Prompt:", prompt)
print("\nGenerated text:")
print(result[0]["generated_text"])

Notice that the output includes the original prompt followed by the model's continuation. The model doesn't "understand" the prompt in a human sense -- it predicts statistically likely continuations based on patterns learned during training.

### Generating Multiple Variations

Because sampling introduces randomness, the same prompt can produce different outputs each time. Setting `num_return_sequences > 1` lets you generate multiple variations in a single call, which is useful for picking the best one.

In [ ]:
# Generate 3 different continuations
prompt = "The future of artificial intelligence is"

results = generator(
    prompt,
    max_length=40,
    num_return_sequences=3,
    temperature=0.8,
    do_sample=True,
    pad_token_id=generator.tokenizer.eos_token_id
)

print(f"Prompt: '{prompt}'\n")
for i, result in enumerate(results, 1):
    print(f"Variation {i}: {result['generated_text']}\n")

---

## Method 2: Using Model and Tokenizer Directly (Advanced)

The pipeline is convenient, but sometimes you need finer control -- for example, to inspect token probabilities, manipulate attention masks, or integrate generation into a larger system. Here we load the tokenizer and model separately.

This two-step process mirrors what the pipeline does internally:
1. **Tokenizer**: Converts text to token IDs (and back)
2. **Model**: Takes token IDs and produces next-token predictions

In [ ]:
# Load tokenizer and model separately
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)

print(f"Model loaded on: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

With the model and tokenizer loaded, we can now control every aspect of the generation process. The key parameters on `model.generate()` are:

- **`max_length`**: Total token limit (prompt + generation)
- **`temperature`**: Scales the logits before sampling. Lower = more deterministic, higher = more random
- **`top_k`**: Only sample from the top K most probable tokens
- **`top_p`** (nucleus sampling): Only sample from the smallest set of tokens whose cumulative probability exceeds P
- **`do_sample`**: If False, uses greedy decoding (always picks the most likely token)

In [ ]:
# Generate text with full control over parameters
prompt = "In the year 2050, technology will"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = model.generate(
    inputs.input_ids,
    max_length=60,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"\nGenerated: {generated_text}")

---

## Understanding Generation Parameters

The quality and style of generated text depends heavily on the decoding parameters. The most important one is **temperature**, which controls the trade-off between coherence and creativity.

Mathematically, temperature $\tau$ rescales the model's output logits $z_i$ before applying softmax:

$$P(x_i) = \frac{\exp(z_i / \tau)}{\sum_j \exp(z_j / \tau)}$$

- $\tau \to 0$: Distribution becomes peaked at the highest-probability token (greedy)
- $\tau = 1$: Uses the model's raw probabilities
- $\tau > 1$: Flattens the distribution, giving more chance to unlikely tokens

Let's see this in practice by generating text at three different temperatures from the same prompt.

In [ ]:
def compare_temperatures(
    prompt: str,
    temperatures: list[float] | None = None,
    max_length: int = 40,
) -> pd.DataFrame:
    """Generate text at different temperatures and compare results.

    Args:
        prompt: The input text to continue.
        temperatures: List of temperature values to compare.
            Defaults to [0.3, 0.7, 1.2].
        max_length: Maximum total length of generated text.

    Returns:
        DataFrame with temperature, description, and generated text.
    """
    if temperatures is None:
        temperatures = [0.3, 0.7, 1.2]

    descriptions = {
        0.3: "Conservative (focused, may repeat)",
        0.7: "Balanced (coherent, some variety)",
        1.2: "Creative (surprising, may diverge)",
    }

    rows: list[dict[str, str | float]] = []
    for temp in temperatures:
        set_seed(SEED)
        result = generator(
            prompt,
            max_length=max_length,
            temperature=temp,
            do_sample=True,
            num_return_sequences=1,
            pad_token_id=generator.tokenizer.eos_token_id,
        )
        generated = result[0]["generated_text"][len(prompt):].strip()
        rows.append({
            "Temperature": temp,
            "Style": descriptions.get(temp, ""),
            "Generated Continuation": generated,
        })

    return pd.DataFrame(rows)


print("Prompt: 'The secret to happiness is'\n")
temp_df = compare_temperatures("The secret to happiness is")
temp_df

The table above shows the trade-off clearly. At low temperature the model sticks to safe, high-probability continuations (which can become repetitive). At high temperature it takes more risks, producing more surprising text that may lose coherence. A temperature around 0.7 is a common default.

### Comparing Decoding Strategies

Beyond temperature, the **decoding strategy** determines *how* the next token is selected from the probability distribution. Let's compare the three most common strategies:

- **Greedy**: Always pick the most likely token. Fast but repetitive.
- **Top-k sampling**: Sample from the top K most likely tokens.
- **Nucleus (top-p) sampling**: Sample from the smallest set of tokens whose probabilities sum to at least P.

In [ ]:
def compare_decoding_strategies(
    prompt: str,
    max_length: int = 50,
) -> pd.DataFrame:
    """Generate text using different decoding strategies and compare.

    Args:
        prompt: The input text to continue.
        max_length: Maximum total length of generated text.

    Returns:
        DataFrame comparing greedy, top-k, and nucleus sampling.
    """
    strategies = [
        {"name": "Greedy", "do_sample": False},
        {"name": "Top-k (k=50)", "do_sample": True, "top_k": 50, "temperature": 0.7},
        {"name": "Nucleus (p=0.95)", "do_sample": True, "top_p": 0.95, "temperature": 0.7},
        {"name": "Top-k + Nucleus", "do_sample": True, "top_k": 50, "top_p": 0.95, "temperature": 0.7},
    ]

    rows: list[dict[str, str]] = []
    for strategy in strategies:
        set_seed(SEED)
        gen_kwargs = {k: v for k, v in strategy.items() if k != "name"}
        result = generator(
            prompt,
            max_length=max_length,
            num_return_sequences=1,
            pad_token_id=generator.tokenizer.eos_token_id,
            **gen_kwargs,
        )
        generated = result[0]["generated_text"][len(prompt):].strip()
        rows.append({"Strategy": strategy["name"], "Generated Continuation": generated})

    return pd.DataFrame(rows)


print("Prompt: 'The most important invention of the 21st century is'\n")
strategy_df = compare_decoding_strategies("The most important invention of the 21st century is")
strategy_df

In practice, **top-k + nucleus sampling** with `temperature=0.7` is the most widely used setting. Greedy decoding is deterministic and fast, but tends to produce repetitive loops in longer sequences.

---

## Practical Applications

Now that we understand the mechanics, let's see text generation applied to realistic scenarios. These examples show how the same model can serve different purposes depending on the prompt and parameter choices.

### Example 1: Story Beginning Generator

Creative writing is one of the most natural fits for text generation. Higher temperatures work well here since we want variety and surprise.

In [ ]:
story_prompts = [
    "Once upon a time in a small village,",
    "The detective looked at the evidence and realized",
    "On the first day of summer vacation,",
]

for prompt in story_prompts:
    set_seed(SEED)
    result = generator(
        prompt,
        max_length=60,
        temperature=0.8,
        do_sample=True,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    print(f"Prompt: {prompt}")
    print(f"Story: {result[0]['generated_text']}\n")

### Example 2: Technical Writing Assistant

For technical or factual prompts, lower temperature keeps the output more focused and less likely to hallucinate. Note that small models like distilgpt2 are not reliable for factual accuracy -- this is just to demonstrate the pattern.

In [ ]:
technical_prompts = [
    "This function calculates the",
    "The main advantage of using a database is",
    "To install the software, first",
]

for prompt in technical_prompts:
    set_seed(SEED)
    result = generator(
        prompt,
        max_length=40,
        temperature=0.5,
        do_sample=True,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    print(f"Prompt: {prompt}")
    print(f"Output: {result[0]['generated_text']}\n")

---

## Performance Benchmarking

Understanding generation speed is important for production use cases. Let's measure how long generation takes across different output lengths, and collect the results into a structured comparison.

In [ ]:
def benchmark_generation(
    prompt: str,
    lengths: list[int] | None = None,
    num_runs: int = 3,
) -> pd.DataFrame:
    """Benchmark text generation speed across different output lengths.

    Runs each length multiple times and reports the average to reduce
    noise from system variance.

    Args:
        prompt: The input text to continue.
        lengths: List of max_length values to benchmark.
            Defaults to [25, 50, 100].
        num_runs: Number of runs to average per length.

    Returns:
        DataFrame with max_length, avg time, and tokens per second.
    """
    if lengths is None:
        lengths = [25, 50, 100]

    prompt_tokens = len(generator.tokenizer.encode(prompt))
    rows: list[dict[str, float | int | str]] = []

    for max_len in lengths:
        times: list[float] = []
        for _ in range(num_runs):
            set_seed(SEED)
            start = time.time()
            generator(
                prompt,
                max_length=max_len,
                do_sample=True,
                pad_token_id=generator.tokenizer.eos_token_id,
            )
            times.append(time.time() - start)

        avg_time = np.mean(times)
        generated_tokens = max_len - prompt_tokens
        tokens_per_sec = generated_tokens / avg_time if avg_time > 0 else 0

        rows.append({
            "Max Length": max_len,
            "Generated Tokens": generated_tokens,
            "Avg Time (s)": round(avg_time, 2),
            "Tokens/sec": round(tokens_per_sec, 1),
            "Device": "GPU" if torch.cuda.is_available() else "CPU",
        })

    return pd.DataFrame(rows)


print(f"Benchmarking {MODEL_NAME} on {device}...\n")
bench_df = benchmark_generation("Artificial intelligence is")
bench_df

Generation time scales roughly linearly with the number of tokens, since each token requires a full forward pass through the model. GPU acceleration provides the biggest speedup for longer sequences where the overhead of transferring data to the GPU is amortized.

---

## Error Analysis: Where Small Models Fail

An important part of working with any ML model is understanding its limitations. Small models like distilgpt2 (82M parameters) are great for learning, but they fail in predictable ways. Recognizing these failure modes helps you choose the right model for your task.

Let's test the model on prompts that require different capabilities and see where it struggles.

In [ ]:
def analyze_failure_modes(
    prompts_with_categories: list[tuple[str, str]],
    max_length: int = 50,
) -> pd.DataFrame:
    """Test the model on prompts designed to expose common failure modes.

    Args:
        prompts_with_categories: List of (prompt, expected_capability) tuples.
        max_length: Maximum total length of generated text.

    Returns:
        DataFrame showing category, prompt, and generated output for review.
    """
    rows: list[dict[str, str]] = []
    for prompt, category in prompts_with_categories:
        set_seed(SEED)
        result = generator(
            prompt,
            max_length=max_length,
            temperature=0.7,
            do_sample=True,
            pad_token_id=generator.tokenizer.eos_token_id,
        )
        generated = result[0]["generated_text"][len(prompt):].strip()
        rows.append({
            "Category": category,
            "Prompt": prompt,
            "Generated": generated[:80] + "..." if len(generated) > 80 else generated,
        })

    return pd.DataFrame(rows)


test_prompts = [
    ("The capital of France is", "Factual knowledge"),
    ("2 + 2 =", "Arithmetic"),
    ("Translate to French: Hello, how are you?", "Instruction following"),
    ("List the planets in our solar system:", "Structured output"),
    ("Once upon a time", "Creative writing"),
]

print(f"Testing {MODEL_NAME} failure modes:\n")
failure_df = analyze_failure_modes(test_prompts)
failure_df

**What to notice:**

- **Factual knowledge**: The model may get simple facts right (it was trained on web text) but often fabricates details. It has no "knowledge base" -- only statistical patterns.
- **Arithmetic**: GPT-2-class models cannot reliably do math. They may produce the right answer for very common equations ("2+2=4") but fail on anything non-trivial.
- **Instruction following**: distilgpt2 was not trained to follow instructions. It will continue the text rather than execute the command. Instruction-tuned models (like Llama, Mistral) handle this well.
- **Structured output**: Without instruction tuning, the model doesn't know how to produce lists, JSON, or other structured formats on demand.
- **Creative writing**: This is where small models shine -- they're quite good at producing fluent, varied prose.

The key takeaway: **model size and training data matter**. For tasks beyond text continuation, you need larger or instruction-tuned models (covered in Notebooks 04 and 10).

---

## Exercises

Try these challenges to deepen your understanding:

1. **Temperature sweep**: Call `compare_temperatures()` with temperatures `[0.1, 0.5, 1.0, 1.5, 2.0]`. At what point does the text become incoherent?

2. **Long-form generation**: Generate text with `max_length=200`. Does the output stay coherent, or does it drift off topic? Compare greedy vs sampling.

3. **Domain-specific prompts**: Test the model with prompts from different domains (medical, legal, scientific). How does quality compare to general prompts?

4. **Model comparison**: If you have GPU access, run the benchmark with both `distilgpt2` and `gpt2-medium`. Compare quality and speed.

5. **Beam search**: Try `num_beams=5` with `do_sample=False` in the pipeline. How does the output differ from sampling?

---

## State-of-the-Art Open Models (Reference)

While this notebook focuses on GPT-2 for educational purposes, the field has advanced significantly. Here are the current leading open-source text generation models:

| Model | Developer | Sizes | Strengths | Requirements |
|-------|-----------|-------|-----------|---------------|
| **Llama 3** | Meta | 8B, 70B | Strong general-purpose, multilingual | 16GB+ VRAM (8B) |
| **Mistral / Mixtral** | Mistral AI | 7B, 8x7B | Efficient, outperforms larger models | 16GB+ VRAM |
| **Qwen 2** | Alibaba | 0.5B-72B | Excellent multilingual | Varies |
| **Gemma** | Google | 2B, 7B | Efficient, strong safety | 8GB+ VRAM |
| **Phi-3** | Microsoft | 3.8B-14B | Great performance for size | 8GB+ VRAM |

**Specialized models:**
- **CodeLlama** (Meta): Code generation specialist
- **Zephyr** (HuggingFace): Conversational fine-tuning

These require more hardware but deliver dramatically better quality. See **Notebook 10** (Ollama) to run these models locally, and **Notebooks 04-05** (Fine-tuning) to train your own.

**Where to explore:**
- [Open LLM Leaderboard](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard)
- [HuggingFace Text Generation Models](https://huggingface.co/models?pipeline_tag=text-generation)

---

## Key Takeaways

- **Pipeline API** is the simplest way to use HuggingFace models -- two lines to generate text
- **Temperature** is the most important generation parameter: lower values produce focused text, higher values produce creative text
- **Decoding strategies** (greedy, top-k, nucleus sampling) control how tokens are selected from the probability distribution
- **Small models** (distilgpt2) excel at text continuation but fail at instruction following, math, and factual accuracy
- Models are downloaded once and **cached locally** -- subsequent runs are much faster

## Next Steps

- **Notebook 02: Text Classification** -- Learn to analyze sentiment and categorize text using BERT
- **Notebook 10: Ollama Integration** -- Run larger, more capable models locally
- **Notebook 04 (NLP): Fine-tuning with Unsloth** -- Train your own text generation model

## Resources

- [Transformers Docs: Text Generation](https://huggingface.co/docs/transformers/main_classes/text_generation)
- [How to Generate Text (HuggingFace Blog)](https://huggingface.co/blog/how-to-generate)
- [GPT-2 Model Card](https://huggingface.co/gpt2)